# TabPFN 3.0: Setup und Split Review

Dieses Notebook stellt das neue Studiendesign her, ohne lange TabPFN-Läufe zu starten. Es prüft, ob die chronologischen Splits und stage-preserving Trainingsgrößen plausibel sind.


In [1]:
import os
import time
from getpass import getpass
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

DATA_DIR = Path("../../data/processed")
RESULT_DIR = DATA_DIR / "tabpfn_3_experiments"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
MASTER_PATH = DATA_DIR / "26_cleaned_master_data.pkl"
TARGET = "target_top_10"

FEATURE_COLUMNS = [
    "distance", "vertical_meters", "stage_nr", "team_tier", "age_at_race",
    "rider_bmi", "wind_stability_index", "weather_temp_mean", "weather_temp_trend",
    "weather_rain_prob_mean", "weather_precipitation_mean", "weather_humidity_mean",
    "gradient_final_km", "lag_rider_points_season", "lag_rider_rank_season",
    "lag_race_competitiveness_median", "lag_team_power_index",
]

META_COLUMNS = ["meta_year", "meta_name", "meta_current_team", "meta_race", "stage_nr", "stage_id"]
TARGET_ROWS = [2_000, 4_000, 6_000, 8_000, 10_000, 12_500, 15_000]


In [2]:
def get_best_available_device():
    try:
        import torch

        if torch.cuda.is_available():
            return "cuda"
        if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            return "mps"
    except Exception:
        pass
    return "cpu"


def clean_tabpfn_token(raw_token):
    token = str(raw_token).strip()
    for prefix in ("export TABPFN_TOKEN=", "TABPFN_TOKEN="):
        if token.startswith(prefix):
            token = token[len(prefix):].strip()
    if (token.startswith('"') and token.endswith('"')) or (token.startswith("'") and token.endswith("'")):
        token = token[1:-1].strip()
    return token


def ensure_tabpfn_ready():
    try:
        import tabpfn
        from tabpfn import TabPFNClassifier
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError("TabPFN fehlt. Bitte zuerst ausführen: %pip install -U tabpfn") from exc

    try:
        from packaging.version import Version
        installed = Version(version("tabpfn"))
        if installed < Version("8.0.0"):
            raise RuntimeError(
                f"Installiert ist tabpfn {installed}. Für TabPFN 3 bitte aktualisieren: %pip install -U tabpfn"
            )
    except ImportError:
        print("Hinweis: Paket 'packaging' fehlt; Versionsprüfung wird übersprungen.")

    existing_token = clean_tabpfn_token(os.environ.get("TABPFN_TOKEN", ""))
    if existing_token:
        os.environ["TABPFN_TOKEN"] = existing_token
        print(f"TABPFN_TOKEN ist gesetzt ({len(existing_token)} Zeichen, endet auf ...{existing_token[-4:]}).")
    else:
        raw_token = getpass("TabPFN API Key einfügen; nur den Key, nicht den export-Befehl: ")
        token = clean_tabpfn_token(raw_token)
        if not token:
            raise RuntimeError("TABPFN_TOKEN wurde nicht gesetzt.")
        os.environ["TABPFN_TOKEN"] = token
        print(f"TABPFN_TOKEN wurde gesetzt ({len(token)} Zeichen, endet auf ...{token[-4:]}).")

    os.environ.pop("TABPFN_NO_BROWSER", None)
    print(f"tabpfn-Version: {getattr(tabpfn, '__version__', 'unbekannt')}")
    return TabPFNClassifier


def load_master_data():
    df = pd.read_pickle(MASTER_PATH).copy()
    df["stage_id"] = (
        df["meta_race"].astype(str)
        + "_"
        + df["meta_year"].astype(str)
        + "_ST"
        + df["stage_nr"].astype(str)
    )
    df["target_top_5"] = np.where(df["rank"] <= 5, 1, 0)
    df["target_top_10"] = np.where(df["rank"] <= 10, 1, 0)
    df["target_top_20"] = np.where(df["rank"] <= 20, 1, 0)
    return df


def make_split_bundle(df, mask):
    split = df.loc[mask].copy().reset_index(drop=True)
    return {
        "X": split[FEATURE_COLUMNS].copy(),
        "y": split[TARGET].astype(int).copy(),
        "y_rank": split["rank"].astype(float).copy(),
        "meta": split[META_COLUMNS].copy(),
    }


def load_experiment_splits():
    df = load_master_data()
    return {
        "train_selection": make_split_bundle(df, df["meta_year"] <= 2022),
        "validation": make_split_bundle(df, df["meta_year"] == 2023),
        "test_original": make_split_bundle(df, df["meta_year"].isin([2024, 2025])),
        "train_final": make_split_bundle(df, df["meta_year"] <= 2023),
    }


def summarize_split(name, bundle):
    meta = bundle["meta"]
    y = bundle["y"]
    stage_counts = meta.groupby("stage_id").size()
    return {
        "split": name,
        "rows": len(meta),
        "years": ", ".join(map(str, sorted(meta["meta_year"].unique()))),
        "n_stages": meta["stage_id"].nunique(),
        "stage_rows_min": int(stage_counts.min()),
        "stage_rows_median": float(stage_counts.median()),
        "stage_rows_mean": float(stage_counts.mean()),
        "stage_rows_max": int(stage_counts.max()),
        "target_positive_rate": float(y.mean()),
    }


def make_stage_preserving_subset(X, y, meta, target_rows, seed):
    stage_counts = meta.groupby("stage_id", sort=False).size().reset_index(name="rows")
    rng = np.random.default_rng(seed)
    shuffled_stage_ids = stage_counts["stage_id"].to_numpy(copy=True)
    rng.shuffle(shuffled_stage_ids)

    selected_stage_ids = []
    selected_rows = 0
    row_lookup = stage_counts.set_index("stage_id")["rows"].to_dict()

    for stage_id in shuffled_stage_ids:
        selected_stage_ids.append(stage_id)
        selected_rows += int(row_lookup[stage_id])
        if selected_rows >= target_rows:
            break

    selected_stage_ids = set(selected_stage_ids)
    mask = meta["stage_id"].isin(selected_stage_ids)

    X_sub = X.loc[mask].reset_index(drop=True)
    y_sub = y.loc[mask].reset_index(drop=True)
    meta_sub = meta.loc[mask].reset_index(drop=True)

    validate_stage_preserving_sample(meta, meta_sub, target_rows)

    sample_info = {
        "target_rows": int(target_rows),
        "actual_rows": int(len(meta_sub)),
        "row_overshoot": int(len(meta_sub) - target_rows),
        "n_stages": int(meta_sub["stage_id"].nunique()),
        "seed": int(seed),
        "positive_rate": float(y_sub.mean()),
    }
    return X_sub, y_sub, meta_sub, sample_info


def validate_stage_preserving_sample(source_meta, sample_meta, target_rows):
    if len(sample_meta) < target_rows:
        raise AssertionError(f"Sample hat nur {len(sample_meta)} Zeilen, Ziel war mindestens {target_rows}.")

    source_counts = source_meta.groupby("stage_id").size()
    sample_counts = sample_meta.groupby("stage_id").size()

    mismatched = []
    for stage_id, sample_count in sample_counts.items():
        source_count = int(source_counts.loc[stage_id])
        if int(sample_count) != source_count:
            mismatched.append((stage_id, int(sample_count), source_count))

    if mismatched:
        raise AssertionError(f"Teilweise Stage-Auswahl gefunden: {mismatched[:5]}")
    return True


def predict_positive_proba(model, X, batch_size):
    proba_parts = []
    current_batch_size = batch_size
    start_idx = 0

    while start_idx < len(X):
        stop_idx = min(start_idx + current_batch_size, len(X))
        X_batch = X.iloc[start_idx:stop_idx]
        try:
            proba_parts.append(model.predict_proba(X_batch)[:, 1])
            print(f"Vorhergesagt: {stop_idx:,}/{len(X):,} Zeilen (Batch: {current_batch_size})")
            start_idx = stop_idx
        except Exception as exc:
            name = type(exc).__name__.lower()
            message = str(exc).lower()
            is_oom = "outofmemory" in name or "out of memory" in message
            if not is_oom or current_batch_size <= 10:
                raise
            current_batch_size = max(10, current_batch_size // 2)
            print(f"Speichergrenze erreicht. Neuer Prediction-Batch: {current_batch_size}")

    return np.concatenate(proba_parts)


def build_prediction_frame(meta, y_rank, y_true, proba, proba_col="p_top10_tabpfn"):
    pred = meta.reset_index(drop=True).copy()
    pred["actual_rank"] = y_rank.reset_index(drop=True).astype(float)
    pred["actual_top10"] = y_true.reset_index(drop=True).astype(int)
    pred[proba_col] = proba
    pred["pred_top10_0_50"] = (pred[proba_col] >= 0.50).astype(int)
    pred["rank_within_stage_by_model"] = (
        pred.groupby("stage_id")[proba_col]
        .rank(ascending=False, method="first")
        .astype(int)
    )
    return pred


def get_stage_top10_predictions(pred, proba_col="p_top10_tabpfn"):
    return (
        pred.sort_values(["stage_id", proba_col], ascending=[True, False])
        .groupby("stage_id", as_index=False)
        .head(10)
        .reset_index(drop=True)
    )


def evaluate_prediction_frame(pred, proba_col="p_top10_tabpfn"):
    y_true = pred["actual_top10"].astype(int)
    proba = pred[proba_col].astype(float)
    hard = (proba >= 0.50).astype(int)

    stage_top10 = get_stage_top10_predictions(pred, proba_col)
    actual_top10_per_stage = pred.groupby("stage_id")["actual_top10"].sum().rename("actual_top10_count")
    stage_eval = (
        stage_top10.groupby("stage_id")
        .agg(
            actual_top10_in_predicted_top10=("actual_top10", "sum"),
            winner_in_predicted_top10=("actual_rank", lambda s: bool((s == 1).any())),
        )
        .join(actual_top10_per_stage)
        .reset_index()
    )
    stage_eval["stage_top10_recall"] = (
        stage_eval["actual_top10_in_predicted_top10"] / stage_eval["actual_top10_count"].replace(0, np.nan)
    )

    return {
        "rows": int(len(pred)),
        "n_stages": int(pred["stage_id"].nunique()),
        "roc_auc": float(roc_auc_score(y_true, proba)),
        "average_precision": float(average_precision_score(y_true, proba)),
        "precision_at_0_50": float(precision_score(y_true, hard, zero_division=0)),
        "recall_at_0_50": float(recall_score(y_true, hard, zero_division=0)),
        "mean_stage_top10_recall": float(stage_eval["stage_top10_recall"].mean()),
        "winner_hit_rate_in_predicted_top10": float(stage_eval["winner_in_predicted_top10"].mean()),
    }


def evaluate_by_year(pred, proba_col="p_top10_tabpfn"):
    rows = []
    for year, pred_year in pred.groupby("meta_year"):
        metrics = evaluate_prediction_frame(pred_year.copy(), proba_col)
        metrics["year"] = int(year)
        rows.append(metrics)
    return pd.DataFrame(rows).sort_values("year")


## Neues Studiendesign

- Training für Modellauswahl: `meta_year <= 2022`
- Validierung: `meta_year == 2023`
- Finaler Test: originaler Zukunftstest `meta_year in [2024, 2025]`
- Finales Training nach Auswahl: `meta_year <= 2023`

Die Sampling-Zielgrößen sind Mindestwerte. Weil vollständige `stage_id`s erhalten bleiben, kann die tatsächliche Zeilenzahl leicht darüber liegen.


In [3]:
splits = load_experiment_splits()

split_overview = pd.DataFrame([
    summarize_split(name, bundle)
    for name, bundle in splits.items()
])

split_overview.to_csv(RESULT_DIR / "split_overview_train2022_val2023_test2024_2025.csv", index=False)
display(split_overview)


,split,rows,years,n_stages,stage_rows_min,stage_rows_median,stage_rows_mean,stage_rows_max,target_positive_rate
0,train_selection,169349,"2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012...",997,117,171.0,169.858576,206,0.059085
1,validation,8897,2023,57,122,158.0,156.087719,175,0.064067
2,test_original,17802,"2024, 2025",112,131,161.5,158.946429,179,0.062016
3,train_final,178246,"2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012...",1054,117,170.0,169.113852,206,0.059334


## Stage-preserving Sampling Dry-Run

Der folgende Dry-Run zeigt, wie viele vollständige Stages pro Zielgröße gezogen werden. Es wird nicht trainiert.


In [4]:
dry_run_rows = []
for split_name in ["train_selection", "train_final"]:
    bundle = splits[split_name]
    for target_rows in TARGET_ROWS:
        _, y_sub, meta_sub, sample_info = make_stage_preserving_subset(
            bundle["X"],
            bundle["y"],
            bundle["meta"],
            target_rows=target_rows,
            seed=42,
        )
        sample_info["split"] = split_name
        sample_info["stage_rows_min"] = int(meta_sub.groupby("stage_id").size().min())
        sample_info["stage_rows_max"] = int(meta_sub.groupby("stage_id").size().max())
        dry_run_rows.append(sample_info)

sampling_plan = pd.DataFrame(dry_run_rows)[[
    "split", "target_rows", "actual_rows", "row_overshoot", "n_stages",
    "stage_rows_min", "stage_rows_max", "positive_rate", "seed",
]]
sampling_plan.to_csv(RESULT_DIR / "stage_preserving_sampling_dry_run.csv", index=False)
display(sampling_plan)


,split,target_rows,actual_rows,row_overshoot,n_stages,stage_rows_min,stage_rows_max,positive_rate,seed
0,train_selection,2000,2041,41,12,136,191,0.059775,42
1,train_selection,4000,4086,86,24,136,195,0.059471,42
2,train_selection,6000,6093,93,36,136,198,0.059741,42
3,train_selection,8000,8118,118,48,131,198,0.059744,42
4,train_selection,10000,10100,100,60,131,198,0.059901,42
5,train_selection,12500,12573,73,75,131,198,0.060049,42
6,train_selection,15000,15009,9,90,131,198,0.060231,42
7,train_final,2000,2094,94,12,132,194,0.056829,42
8,train_final,4000,4017,17,23,132,198,0.057506,42
9,train_final,6000,6075,75,35,132,198,0.057778,42


## Checks

Diese Checks sollten ohne Fehler laufen. Sie bestätigen, dass die Jahre korrekt getrennt sind und die Stage-Samples keine Etappen zerschneiden.


In [5]:
assert splits["train_selection"]["meta"]["meta_year"].max() <= 2022
assert set(splits["validation"]["meta"]["meta_year"].unique()) == {2023}
assert set(splits["test_original"]["meta"]["meta_year"].unique()) == {2024, 2025}
assert splits["train_final"]["meta"]["meta_year"].max() <= 2023
assert sampling_plan["actual_rows"].ge(sampling_plan["target_rows"]).all()

print("Alle Split- und Stage-Preserving-Checks erfolgreich.")


Alle Split- und Stage-Preserving-Checks erfolgreich.
